# Two-tower training on Colab (Phase 5)

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

Data comes from the Azure ML data asset `hm-features:1`; every run logs to the Azure ML workspace `mlw-twotower` (experiment `phase5-training`).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code and install the extra packages
Torch, numpy and pandas are preinstalled on Colab; only Azure/MLflow packages are added.

In [ ]:
import os
REPO = '/content/TwoTower_RecSys'
if not os.path.exists(REPO):
    !git clone -q https://github.com/kumarbaibhav6/TwoTower_RecSys.git {REPO}
%cd {REPO}
!git pull -q && git log --oneline -1
%pip install -q -r requirements-colab.txt

## 3. Sign in to Azure
Installs the Azure CLI (~1 min), then prints a code: open https://microsoft.com/devicelogin, enter it and sign in with the account that owns the subscription. Both the data download and MLflow logging use this login.

In [ ]:
!curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash > /dev/null
!az login --use-device-code --tenant 83815c58-2b1f-42d0-ab73-3653d9dd6ff6 -o none
!az account set --subscription e00b6cd4-18c0-4e2d-87d9-432fef357d37 && az account show --query name -o tsv

## 4. Download the features (~500 MB)

In [ ]:
!python -m infra.download_asset hm-features:1 data/features/full
!ls -la data/features/full

## 5. Main run
All features, logQ on, accidental-hit masking on. Early stopping on validation MAP@12.

In [ ]:
!python -m src.train --features-dir data/features/full --run-name full-main --epochs 8 --num-workers 2

## 6. Ablations
Each run switches off one component. Fewer epochs and 20K sampled val users per epoch keep the total inside a Colab session; final val/test numbers still use all users.

In [ ]:
ablations = {
    'full-no-logq': '--no-logq',
    'full-no-history': '--no-history',
    'full-no-text': '--no-text',
    'full-no-user-id': '--no-user-id',
}
for name, flag in ablations.items():
    print('=' * 20, name)
    !python -m src.train --features-dir data/features/full --run-name {name} {flag} --epochs 4 --val-max-users 20000 --num-workers 2

## 7. Results
The JSON for every run is also saved as an MLflow artifact, so it can be pulled into the repo from the laptop.

In [ ]:
import json, glob
for path in sorted(glob.glob('benchmarks/train_full-*.json')):
    r = json.load(open(path))
    v, t = r['val'], r['test']
    print(f"{r['run_name']:18s} best epoch {r['best_epoch']} | val MAP@12 {v['map@12']:.4f} R@100 {v['recall@100']:.4f} | test MAP@12 {t['map@12']:.4f} R@100 {t['recall@100']:.4f}")